# 01 — Fetch & Save Data
Fetch January 2024 actuals and forecasts from Elexon BMRS API and save to CSV.

In [ ]:
import requests
import pandas as pd
import os

BASE = 'https://data.elexon.co.uk/bmrs/api/v1'
os.makedirs('data', exist_ok=True)

In [ ]:
# Fetch actuals — FUELHH, filter WIND, full January 2024
print('Fetching actuals...')
url = f'{BASE}/datasets/FUELHH/stream?settlementDateFrom=2024-01-01&settlementDateTo=2024-01-31'
resp = requests.get(url, timeout=60)
resp.raise_for_status()
data = resp.json()
print(f'Total FUELHH records: {len(data)}')

actuals = pd.DataFrame([d for d in data if d.get('fuelType') == 'WIND'])
actuals['startTime'] = pd.to_datetime(actuals['startTime'], utc=True)
actuals = actuals[['startTime', 'generation']].sort_values('startTime').reset_index(drop=True)
print(f'WIND actuals: {len(actuals)} rows')
print(actuals.head())
actuals.to_csv('data/actuals_jan2024.csv', index=False)
print('Saved to data/actuals_jan2024.csv')

In [ ]:
# Fetch forecasts — WINDFOR
# publishDateTimeFrom goes 48h before Jan 1 to capture long-horizon forecasts
print('Fetching forecasts...')
url = f'{BASE}/datasets/WINDFOR/stream?publishDateTimeFrom=2023-12-30T00:00:00Z&publishDateTimeTo=2024-01-31T23:59:59Z'
resp = requests.get(url, timeout=120)
resp.raise_for_status()
data = resp.json()
print(f'Total WINDFOR records: {len(data)}')

forecasts = pd.DataFrame(data)
forecasts['startTime'] = pd.to_datetime(forecasts['startTime'], utc=True)
forecasts['publishTime'] = pd.to_datetime(forecasts['publishTime'], utc=True)

# Only keep rows where startTime is in Jan 2024
jan_start = pd.Timestamp('2024-01-01', tz='UTC')
jan_end   = pd.Timestamp('2024-02-01', tz='UTC')
forecasts = forecasts[(forecasts['startTime'] >= jan_start) & (forecasts['startTime'] < jan_end)]

# Compute horizon and keep 0-48h only
forecasts['horizon_h'] = (forecasts['startTime'] - forecasts['publishTime']).dt.total_seconds() / 3600
forecasts = forecasts[(forecasts['horizon_h'] >= 0) & (forecasts['horizon_h'] <= 48)]
forecasts = forecasts[['startTime', 'publishTime', 'generation', 'horizon_h']].sort_values(['startTime','publishTime']).reset_index(drop=True)

print(f'Filtered forecasts: {len(forecasts)} rows')
print(forecasts.head())
forecasts.to_csv('data/forecasts_jan2024.csv', index=False)
print('Saved to data/forecasts_jan2024.csv')

In [ ]:
# Quick sanity check
print('=== ACTUALS ===')
print(f'Rows: {len(actuals)}')
print(f'Date range: {actuals.startTime.min()} → {actuals.startTime.max()}')
print(f'Generation range: {actuals.generation.min()} → {actuals.generation.max()} MW')
print(f'Nulls: {actuals.isnull().sum().sum()}')

print('\n=== FORECASTS ===')
print(f'Rows: {len(forecasts)}')
print(f'startTime range: {forecasts.startTime.min()} → {forecasts.startTime.max()}')
print(f'Horizon range: {forecasts.horizon_h.min():.1f}h → {forecasts.horizon_h.max():.1f}h')
print(f'Unique publish times: {forecasts.publishTime.nunique()}')
print(f'Nulls: {forecasts.isnull().sum().sum()}')